In [ ]:
#!/usr/bin/env python3
"""
Fit DDM and SRDM to rr98 with configurable difficulty resolution.

   ╔══════════════════════════════════════════╗
   ║  CHANGE THIS ONE NUMBER AND RUN AGAIN:  ║
   ║                                         ║
   ║      N_LEVELS = 7   (or 11, 16, 33)     ║
   ╚══════════════════════════════════════════╝

  - 7  = the original qcut binning (fewest params, most trials per bin)
  - 11 = moderate resolution
  - 16 = one level per 2 strength steps
  - 33 = one level per raw strength value (most params, fewest trials per bin)

Uses physical-brightness correctness (strength > 16 = bright).
Excludes strength == 16 (exactly ambiguous).
Fits both DDM and SRDM (c_only) for all 3 participants via optimize().

OUTPUT FILES (one per model, not one per participant/N_LEVELS!):
  - fits_ddm.csv   -- one row per (participant, n_levels) fit
  - fits_srdm.csv  -- one row per (participant, n_levels) fit
Re-running with a different N_LEVELS (or a different participant subset)
appends/updates rows in these same two files rather than creating new
files, so all your grouping-resolution comparisons end up living in
one place per model.
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

# ╔═══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these                              ║
# ╚═══════════════════════════════════════════════════════════╝
N_LEVELS     = 16       # <-- CHANGE THIS: 7, 11, 16, 33, etc.
DATA_PATH    = "../../rr98.csv"
STAN_DDM     = "DDM_rr98_configurable.stan"
STAN_SRDM    = "SRDM_rr98_c_only_configurable.stan"
STAN_RDM    = "SRDM_rr98_B_only_configurable.stan"
PARTICIPANTS = ["jf", "kr", "nh"]

DDM_OUT  = "fits_ddm.csv"
SRDM_OUT = "fits_srdm.csv"


# ═══════════════════════════════════════════════════════════
# Data prep
# ═══════════════════════════════════════════════════════════
def load_data():
    df = pd.read_csv(DATA_PATH)
    df = df[df["outlier"] == False].copy()
    df["correct"] = df["correct"].astype(int)
    df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

    # Physical correctness
    df = df[df["strength"] != 16].copy()
    df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

    # Bin strength into N_LEVELS groups
    if N_LEVELS == 33:
        # Special case: use raw strength directly (0-15 -> 1-16, 17-32 -> 17-32)
        # But strength=16 is already excluded, so we have 32 values.
        # Map to 1..32 (not 33) since strength=16 is gone.
        unique_strengths = sorted(df["strength"].unique())
        strength_to_level = {s: i+1 for i, s in enumerate(unique_strengths)}
        df["diff_level"] = df["strength"].map(strength_to_level)
        actual_levels = len(unique_strengths)
    else:
        def _qcut_levels(s):
            return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
        df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
        actual_levels = df["diff_level"].nunique()

    df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]

    print(f"N_LEVELS requested: {N_LEVELS}, actual unique levels: {actual_levels}")
    print(f"Trials: {len(df)}, cells: {df['cell'].nunique()} (2 x {actual_levels})")
    print(f"Trials per level (min/median/max): "
          f"{df.groupby(['id','cell']).size().min()} / "
          f"{int(df.groupby(['id','cell']).size().median())} / "
          f"{df.groupby(['id','cell']).size().max()}")

    return df, actual_levels


def build_data(df, pid, n_levels):
    d = df[df["id"] == pid]
    d_correct = d[d["act_correct"] == 1]
    d_false = d[d["act_correct"] == 0]
    max_rt = float(d["rt"].max())
    t0_hi = float(d["rt"].quantile(0.05))
    return {
        "N_LEVELS": n_levels,
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(),
        "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "max_rt": max_rt, "t0_hi": t0_hi,
    }


# ═══════════════════════════════════════════════════════════
# Fitting
# ═══════════════════════════════════════════════════════════
def fit_ddm(model, data):
    nl = data["N_LEVELS"]
    inits = {
        "a": [0.8, 1.5], "v_base": [2.0]*nl,
        "sv": 0.5, "sz": 0.1,
        "t0": 0.2 * data["t0_hi"], "p_lapse": 0.02,
    }
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=True)


def fit_srdm(model, data):
    nl = data["N_LEVELS"]
    inits = {
        "c": [0.0, 0.0], "B": 1.0,
        "t0": 0.5 * data["t0_hi"],
        "d_base": [1.5]*nl, "r": 4.5, "p_lapse": 0.02,
    }
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=True)


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll,
            "BIC": n_params*np.log(n) - 2*total_ll}


# ═══════════════════════════════════════════════════════════
# Consolidated CSV output
# ═══════════════════════════════════════════════════════════
def save_fit_row(csv_path, pid, n_levels_requested, n_levels_actual, mle, ic):
    """
    Append/update a single fit's summary row in a shared per-model CSV.

    - Keeps every fitted (scalar and vector) parameter as its own column.
    - Drops the per-trial log_lik[...] columns (there can be thousands of
      these -- they're only used here to get the total log-likelihood,
      and don't belong in a tidy cross-participant/cross-binning summary
      table). The summed log-lik is kept as `log_lik_total`.
    - If a row already exists for this (pid, n_levels_actual) combo, it's
      replaced, so re-running a fit overwrites cleanly instead of
      duplicating.
    """
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()

    row = {
        "pid": pid,
        "n_levels_requested": n_levels_requested,
        "n_levels_actual": n_levels_actual,
        "n_params": ic["n_params"],
        "n_trials": ic["n_trials"],
        "log_lik_total": ic["log_lik"],
        "AIC": ic["AIC"],
        "BIC": ic["BIC"],
        "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **row,
    }
    new_row = pd.DataFrame([row])

    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask_same = (existing["pid"] == pid) & (existing["n_levels_actual"] == n_levels_actual)
        existing = existing[~mask_same]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row

    combined = combined.sort_values(["n_levels_actual", "pid"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)
    return combined


# ═══════════════════════════════════════════════════════════
# Main
# ═══════════════════════════════════════════════════════════
def main():
    df, actual_levels = load_data()

    # Parameter counts: a[2] + v_or_d[actual_levels] + sv + sz + t0 + p_lapse
    ddm_n_params  = 2 + actual_levels + 4   # a, v_base, sv, sz, t0, p_lapse
    srdm_n_params = 2 + actual_levels + 3   # c, d_base, B, r, t0, p_lapse

    print(f"\nDDM params: {ddm_n_params} ({actual_levels} drift rates)")
    print(f"SRDM params: {srdm_n_params} ({actual_levels} d' values)")

    print("\nCompiling models...")
    ddm_model = CmdStanModel(stan_file=STAN_DDM)
    srdm_model = CmdStanModel(stan_file=STAN_SRDM)

    for pid in PARTICIPANTS:
        print(f"\n{'='*60}")
        print(f"  {pid}  (N_LEVELS={actual_levels})")
        print(f"{'='*60}")

        data = build_data(df, pid, actual_levels)

        # DDM
        print(f"\n  --- DDM ---")
        ddm_mle = fit_ddm(ddm_model, data)
        ic = aic_bic(ddm_mle, ddm_n_params)
        save_fit_row(DDM_OUT, pid, N_LEVELS, actual_levels, ddm_mle, ic)
        row = ddm_mle.optimized_params_pd.iloc[0]
        print(f"  a=[{row['a[1]']:.3f}, {row['a[2]']:.3f}]  t0={row['t0']:.4f}  "
              f"sv={row['sv']:.4f}  sz={row['sz']:.4f}  p_lapse={row['p_lapse']:.4f}")
        print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

        # SRDM
        print(f"\n  --- SRDM ---")
        srdm_mle = fit_srdm(srdm_model, data)
        ic2 = aic_bic(srdm_mle, srdm_n_params)
        save_fit_row(SRDM_OUT, pid, N_LEVELS, actual_levels, srdm_mle, ic2)
        row = srdm_mle.optimized_params_pd.iloc[0]
        print(f"  c=[{row['c[1]']:.3f}, {row['c[2]']:.3f}]  B={row['B']:.3f}  "
              f"r={row['r']:.2f}  t0={row['t0']:.4f}  p_lapse={row['p_lapse']:.4f}")
        print(f"  LL={ic2['log_lik']:.1f}  AIC={ic2['AIC']:.1f}  BIC={ic2['BIC']:.1f}")

        delta = ic["AIC"] - ic2["AIC"]
        print(f"\n  ΔAIC(DDM-SRDM) = {delta:+.1f}  ({'DDM' if delta<0 else 'SRDM'} wins)")

    print(f"\nAll fits written/updated in:\n  {DDM_OUT}\n  {SRDM_OUT}")


if __name__ == "__main__":
    main()


N_LEVELS requested: 16, actual unique levels: 16
Trials: 22584, cells: 32 (2 x 16)
Trials per level (min/median/max): 104 / 209 / 453

DDM params: 22 (16 drift rates)
SRDM params: 21 (16 d' values)

Compiling models...


ValueError: no such file /Users/Anton/Documents/ForskningsprojektSommer2026/RR98fit/DDM/NoLapse/DDM_rr98_configurable.stan

In [ ]:
#!/usr/bin/env python3
"""
Plot fitted parameters from DDM and SRDM MAP estimates, all 3 participants.

Reads from the two consolidated fit files produced by the fitting cell
(fits_ddm.csv / fits_srdm.csv), each of which stacks every participant
and every N_LEVELS run as its own row. Pick which binning resolution to
plot with PLOT_N_LEVELS below (defaults to the finest resolution present).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── EDIT THESE ──────────────────────────────────────────────────────────
DDM_FITS_PATH  = "fits_ddm.csv"
SRDM_FITS_PATH = "fits_srdm.csv"
PLOT_N_LEVELS  = None   # <-- set to e.g. 7, 11, 16, 33 to pick a binning;
                        #     None = use the finest (max) n_levels_actual present

PARTICIPANTS = ["jf", "kr", "nh"]
COLORS = {"jf": "tab:blue", "kr": "tab:orange", "nh": "tab:green"}


def load_model_fits(path, n_levels, vec_prefix):
    """Load one row per participant at a given n_levels_actual from a
    consolidated fits CSV, returning {pid: {param: value_or_list}}."""
    df = pd.read_csv(path)
    df = df[df["n_levels_actual"] == n_levels]

    out = {}
    for _, row in df.iterrows():
        pid = row["pid"]
        vec_cols = sorted(
            [c for c in df.columns if c.startswith(vec_prefix + "[")],
            key=lambda c: int(c.split("[")[1].rstrip("]")),
        )
        out[pid] = dict(row)
        out[pid][vec_prefix] = [row[c] for c in vec_cols]
    return out


def main():
    ddm_df = pd.read_csv(DDM_FITS_PATH)
    srdm_df = pd.read_csv(SRDM_FITS_PATH)

    n_levels = PLOT_N_LEVELS
    if n_levels is None:
        n_levels = int(max(ddm_df["n_levels_actual"].max(),
                            srdm_df["n_levels_actual"].max()))

    ddm = load_model_fits(DDM_FITS_PATH, n_levels, "v_base")
    srdm = load_model_fits(SRDM_FITS_PATH, n_levels, "d_base")

    n_lev_ddm = len(next(iter(ddm.values()))["v_base"]) if ddm else 0
    n_lev_srdm = len(next(iter(srdm.values()))["d_base"]) if srdm else 0
    n_lev = max(n_lev_ddm, n_lev_srdm)
    x = np.arange(1, n_lev + 1)

    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    fig.suptitle(f"Parameter comparison — DDM vs SRDM  ({n_lev} difficulty levels)",
                 fontsize=13, fontweight="bold")

    # ── Row 0: drift / d' curves ──────────────────────────────────────
    ax_v = axes[0, 0]
    ax_v.set_title("DDM: drift rate (v_base)")
    for pid in PARTICIPANTS:
        if pid in ddm:
            ax_v.plot(x[:len(ddm[pid]["v_base"])], ddm[pid]["v_base"],
                      "o-", color=COLORS[pid], ms=3, lw=1.2, label=pid)
    ax_v.set_xlabel("Difficulty level")
    ax_v.set_ylabel("v")
    ax_v.legend(fontsize=8)
    ax_v.axhline(0, color="gray", ls="--", lw=0.5)

    ax_d = axes[0, 1]
    ax_d.set_title("SRDM: discriminability (d_base)")
    for pid in PARTICIPANTS:
        if pid in srdm:
            ax_d.plot(x[:len(srdm[pid]["d_base"])], srdm[pid]["d_base"],
                      "s-", color=COLORS[pid], ms=3, lw=1.2, label=pid)
    ax_d.set_xlabel("Difficulty level")
    ax_d.set_ylabel("d'")
    ax_d.legend(fontsize=8)
    ax_d.axhline(0, color="gray", ls="--", lw=0.5)

    # Overlay both on same axes for direct comparison (normalized)
    ax_both = axes[0, 2]
    ax_both.set_title("Overlay: v (solid) vs d' (dashed)")
    for pid in PARTICIPANTS:
        if pid in ddm:
            v = ddm[pid]["v_base"]
            ax_both.plot(x[:len(v)], v, "-", color=COLORS[pid], lw=1.2, label=f"{pid} v")
        if pid in srdm:
            d = srdm[pid]["d_base"]
            ax2 = ax_both.twinx() if pid == PARTICIPANTS[0] else ax_both.twinx()
            ax2.plot(x[:len(d)], d, "--", color=COLORS[pid], lw=1.2, alpha=0.7)
            if pid == PARTICIPANTS[-1]:
                ax2.set_ylabel("d' (dashed)", fontsize=8)
    ax_both.set_xlabel("Difficulty level")
    ax_both.set_ylabel("v (solid)")
    ax_both.axhline(0, color="gray", ls="--", lw=0.5)

    # ── Row 1: boundary / threshold params ────────────────────────────
    ax_a = axes[1, 0]
    ax_a.set_title("DDM: boundary separation (a)")
    bar_x = np.arange(len(PARTICIPANTS))
    w = 0.35
    a_speed = [ddm[p]["a[1]"] if p in ddm else 0 for p in PARTICIPANTS]
    a_acc   = [ddm[p]["a[2]"] if p in ddm else 0 for p in PARTICIPANTS]
    ax_a.bar(bar_x - w/2, a_speed, w, label="speed", color="skyblue", edgecolor="black", lw=0.5)
    ax_a.bar(bar_x + w/2, a_acc,   w, label="accuracy", color="salmon", edgecolor="black", lw=0.5)
    ax_a.set_xticks(bar_x)
    ax_a.set_xticklabels(PARTICIPANTS)
    ax_a.set_ylabel("a")
    ax_a.legend(fontsize=8)

    ax_c = axes[1, 1]
    ax_c.set_title("SRDM: criterion (c) by SAT")
    c_speed = [srdm[p]["c[1]"] if p in srdm else 0 for p in PARTICIPANTS]
    c_acc   = [srdm[p]["c[2]"] if p in srdm else 0 for p in PARTICIPANTS]
    ax_c.bar(bar_x - w/2, c_speed, w, label="speed", color="skyblue", edgecolor="black", lw=0.5)
    ax_c.bar(bar_x + w/2, c_acc,   w, label="accuracy", color="salmon", edgecolor="black", lw=0.5)
    ax_c.set_xticks(bar_x)
    ax_c.set_xticklabels(PARTICIPANTS)
    ax_c.set_ylabel("c")
    ax_c.axhline(0, color="gray", ls="--", lw=0.5)
    ax_c.legend(fontsize=8)

    ax_Br = axes[1, 2]
    ax_Br.set_title("SRDM: threshold (B) and spike rate (r)")
    B_vals = [srdm[p]["B"] if p in srdm else 0 for p in PARTICIPANTS]
    r_vals = [srdm[p]["r"] if p in srdm else 0 for p in PARTICIPANTS]
    ax_Br.bar(bar_x - w/2, B_vals, w, label="B", color="mediumpurple", edgecolor="black", lw=0.5)
    ax_Br2 = ax_Br.twinx()
    ax_Br2.bar(bar_x + w/2, r_vals, w, label="r", color="gold", edgecolor="black", lw=0.5)
    ax_Br.set_xticks(bar_x)
    ax_Br.set_xticklabels(PARTICIPANTS)
    ax_Br.set_ylabel("B")
    ax_Br2.set_ylabel("r")
    ax_Br.legend(loc="upper left", fontsize=8)
    ax_Br2.legend(loc="upper right", fontsize=8)

    # ── Row 2: t0, sv/sz, lapse ───────────────────────────────────────
    ax_t0 = axes[2, 0]
    ax_t0.set_title("Non-decision time (t0)")
    t0_ddm  = [ddm[p]["t0"]  if p in ddm  else 0 for p in PARTICIPANTS]
    t0_srdm = [srdm[p]["t0"] if p in srdm else 0 for p in PARTICIPANTS]
    ax_t0.bar(bar_x - w/2, t0_ddm,  w, label="DDM",  color="navy", alpha=0.7, edgecolor="black", lw=0.5)
    ax_t0.bar(bar_x + w/2, t0_srdm, w, label="SRDM", color="darkorange", alpha=0.7, edgecolor="black", lw=0.5)
    ax_t0.set_xticks(bar_x)
    ax_t0.set_xticklabels(PARTICIPANTS)
    ax_t0.set_ylabel("t0 (s)")
    ax_t0.legend(fontsize=8)

    ax_sv = axes[2, 1]
    ax_sv.set_title("DDM: sv and sz")
    sv_vals = [ddm[p]["sv"] if p in ddm else 0 for p in PARTICIPANTS]
    sz_vals = [ddm[p]["sz"] if p in ddm else 0 for p in PARTICIPANTS]
    ax_sv.bar(bar_x - w/2, sv_vals, w, label="sv", color="teal", edgecolor="black", lw=0.5)
    ax_sv.bar(bar_x + w/2, sz_vals, w, label="sz", color="coral", edgecolor="black", lw=0.5)
    ax_sv.set_xticks(bar_x)
    ax_sv.set_xticklabels(PARTICIPANTS)
    ax_sv.legend(fontsize=8)
    # Reference line for R&R98 sv value (rescaled to s=1)
    ax_sv.axhline(0.63, color="teal", ls=":", lw=1, alpha=0.5)
    ax_sv.text(0.02, 0.95, "R&R98 η≈0.63", transform=ax_sv.transAxes,
               fontsize=7, color="teal", va="top")

    ax_lapse = axes[2, 2]
    ax_lapse.set_title("Lapse rate (p_lapse)")
    lp_ddm  = [ddm[p]["p_lapse"]  if p in ddm  else 0 for p in PARTICIPANTS]
    lp_srdm = [srdm[p]["p_lapse"] if p in srdm else 0 for p in PARTICIPANTS]
    ax_lapse.bar(bar_x - w/2, lp_ddm,  w, label="DDM",  color="navy", alpha=0.7, edgecolor="black", lw=0.5)
    ax_lapse.bar(bar_x + w/2, lp_srdm, w, label="SRDM", color="darkorange", alpha=0.7, edgecolor="black", lw=0.5)
    ax_lapse.set_xticks(bar_x)
    ax_lapse.set_xticklabels(PARTICIPANTS)
    ax_lapse.set_ylabel("p_lapse")
    ax_lapse.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"params_{n_lev}lev.png", dpi=140, bbox_inches="tight")
    print(f"Saved: params_{n_lev}lev.png")
    plt.show()


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
"""
Fit DDM and SRDM to rr98 WITHOUT a lapse mixture.

Instead of a lapse process, trials are trimmed (fastest/slowest 1% per
participant x instruction x difficulty cell removed, Ratcliff 2008 style)
and t0 is hard-bounded above by the fastest surviving RT for that
participant (t0_hi = min(RT) over the trimmed data). No blend, no
p_lapse, no mixture -- the model likelihood is evaluated directly.

Uses physical-brightness correctness (strength > 16 = bright).
Excludes strength == 16 (exactly ambiguous), same as the lapse version.

OUTPUT FILES (one per model, all participants stacked, same convention
as the lapse-model fitting cell):
  - fits_ddm_nolapse.csv
  - fits_srdm_nolapse.csv
"""

import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from datetime import datetime, timezone

import numpy as np
import pandas as pd
from cmdstanpy import CmdStanModel

# ╔═══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these                              ║
# ╚═══════════════════════════════════════════════════════════╝
N_LEVELS      = 16      # <-- CHANGE THIS: 7, 11, 16, 33, etc. (16 was the
                        #     BIC-best resolution found with the lapse models)
DATA_PATH     = "../../rr98.csv"
STAN_DDM      = "DDM_rr98_nolapse.stan"
STAN_SRDM     = "SRDM_rr98_c_only_nolapse.stan"
PARTICIPANTS  = ["jf", "kr", "nh"]
TRIM_LOW      = 0.01    # trim fastest 1%
TRIM_HIGH     = 0.99    # trim slowest 1% (i.e. keep [1st, 99th] percentile)

DDM_OUT  = "fits_ddm_nolapse.csv"
SRDM_OUT = "fits_srdm_nolapse.csv"


# ═══════════════════════════════════════════════════════════
# Data prep
# ═══════════════════════════════════════════════════════════
def load_data():
    df = pd.read_csv(DATA_PATH)
    df = df[df["outlier"] == False].copy()
    df["correct"] = df["correct"].astype(int)
    df["sat_id"] = df["instruction"].map({"speed": 1, "accuracy": 2})

    # Physical correctness
    df = df[df["strength"] != 16].copy()
    df["act_correct"] = ((df["strength"] > 16) == (df["response"] == "light")).astype(int)

    # Bin strength into N_LEVELS groups (identical logic to the lapse version)
    if N_LEVELS == 33:
        unique_strengths = sorted(df["strength"].unique())
        strength_to_level = {s: i+1 for i, s in enumerate(unique_strengths)}
        df["diff_level"] = df["strength"].map(strength_to_level)
        actual_levels = len(unique_strengths)
    else:
        def _qcut_levels(s):
            return pd.qcut(s, q=N_LEVELS, labels=False, duplicates="drop") + 1
        df["diff_level"] = df.groupby("id")["strength"].transform(_qcut_levels)
        actual_levels = df["diff_level"].nunique()

    df["cell"] = (df["sat_id"] - 1) * actual_levels + df["diff_level"]

    print(f"N_LEVELS requested: {N_LEVELS}, actual unique levels: {actual_levels}")
    print(f"Trials before trimming: {len(df)}")

    return df, actual_levels


def trim_extremes(df, low=TRIM_LOW, high=TRIM_HIGH):
    """Trim the fastest/slowest tails within each (id, cell) group, i.e.
    per participant x instruction x difficulty cell -- not globally."""
    def _trim_group(g):
        lo, hi = g["rt"].quantile([low, high])
        return g[(g["rt"] >= lo) & (g["rt"] <= hi)]

    trimmed = df.groupby(["id", "cell"], group_keys=False).apply(_trim_group)
    print(f"Trials after trimming ({low:.0%}/{high:.0%} per id x cell): "
          f"{len(trimmed)}  (dropped {len(df) - len(trimmed)}, "
          f"{(1 - len(trimmed)/len(df)):.1%})")
    return trimmed


def build_data(df, pid, n_levels):
    d = df[df["id"] == pid]
    d_correct = d[d["act_correct"] == 1]
    d_false = d[d["act_correct"] == 0]
    t0_hi = float(d["rt"].min())   # hard bound: fastest TRIMMED RT for this pid
    return {
        "N_LEVELS": n_levels,
        "N_correct": len(d_correct), "N_false": len(d_false),
        "rt_correct": d_correct["rt"].to_numpy(),
        "rt_false": d_false["rt"].to_numpy(),
        "cell_correct": d_correct["cell"].to_numpy(dtype=int),
        "cell_false": d_false["cell"].to_numpy(dtype=int),
        "t0_hi": t0_hi,
    }


# ═══════════════════════════════════════════════════════════
# Fitting
# ═══════════════════════════════════════════════════════════
def fit_ddm(model, data):
    nl = data["N_LEVELS"]
    inits = {
        "a": [0.8, 1.5], "v_base": [2.0]*nl,
        "sv": 0.5, "sz": 0.1,
        "t0": 0.2 * data["t0_hi"],
    }
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=True)


def fit_srdm(model, data):
    nl = data["N_LEVELS"]
    inits = {
        "c": [0.0, 0.0], "B": 1.0,
        "t0": 0.5 * data["t0_hi"],
        "d_base": [1.5]*nl, "r": 4.5,
    }
    return model.optimize(data=data, inits=inits, algorithm="lbfgs",
                          iter=5000, show_console=True)


def aic_bic(mle, n_params):
    p = mle.optimized_params_pd
    ll_cols = [c for c in p.columns if c.startswith("log_lik")]
    total_ll = p[ll_cols].iloc[0].sum()
    n = len(ll_cols)
    return {"n_params": n_params, "n_trials": n, "log_lik": total_ll,
            "AIC": 2*n_params - 2*total_ll,
            "BIC": n_params*np.log(n) - 2*total_ll}


# ═══════════════════════════════════════════════════════════
# Consolidated CSV output (same convention as the lapse fitting cell)
# ═══════════════════════════════════════════════════════════
def save_fit_row(csv_path, pid, n_levels_requested, n_levels_actual, mle, ic,
                  extra=None):
    raw = mle.optimized_params_pd.iloc[0]
    keep_cols = [c for c in raw.index if not c.startswith("log_lik")]
    row = raw[keep_cols].to_dict()

    row = {
        "pid": pid,
        "n_levels_requested": n_levels_requested,
        "n_levels_actual": n_levels_actual,
        "n_params": ic["n_params"],
        "n_trials": ic["n_trials"],
        "log_lik_total": ic["log_lik"],
        "AIC": ic["AIC"],
        "BIC": ic["BIC"],
        "fit_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        **(extra or {}),
        **row,
    }
    new_row = pd.DataFrame([row])

    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        mask_same = (existing["pid"] == pid) & (existing["n_levels_actual"] == n_levels_actual)
        existing = existing[~mask_same]
        combined = pd.concat([existing, new_row], ignore_index=True, sort=False)
    else:
        combined = new_row

    combined = combined.sort_values(["n_levels_actual", "pid"]).reset_index(drop=True)
    combined.to_csv(csv_path, index=False)
    return combined


# ═══════════════════════════════════════════════════════════
# Main
# ═══════════════════════════════════════════════════════════
def main():
    df, actual_levels = load_data()
    df = trim_extremes(df)

    # Parameter counts (no p_lapse in the no-lapse models):
    #   DDM:  a[2] + v_base[levels] + sv + sz + t0        = 2 + levels + 3
    #   SRDM: c[2] + d_base[levels] + B + r + t0          = 2 + levels + 3
    ddm_n_params  = 2 + actual_levels + 3
    srdm_n_params = 2 + actual_levels + 3

    print(f"\nDDM params: {ddm_n_params} ({actual_levels} drift rates, no lapse)")
    print(f"SRDM params: {srdm_n_params} ({actual_levels} d' values, no lapse)")

    print("\nCompiling models...")
    ddm_model = CmdStanModel(stan_file=STAN_DDM)
    srdm_model = CmdStanModel(stan_file=STAN_SRDM)

    for pid in PARTICIPANTS:
        print(f"\n{'='*60}")
        print(f"  {pid}  (N_LEVELS={actual_levels}, no lapse, trimmed)")
        print(f"{'='*60}")

        data = build_data(df, pid, actual_levels)
        print(f"  N_correct={data['N_correct']}  N_false={data['N_false']}  "
              f"t0_hi (min trimmed RT)={data['t0_hi']:.4f}")

        # DDM
        print(f"\n  --- DDM (no lapse) ---")
        ddm_mle = fit_ddm(ddm_model, data)
        ic = aic_bic(ddm_mle, ddm_n_params)
        save_fit_row(DDM_OUT, pid, N_LEVELS, actual_levels, ddm_mle, ic,
                     extra={"t0_hi": data["t0_hi"]})
        row = ddm_mle.optimized_params_pd.iloc[0]
        print(f"  a=[{row['a[1]']:.3f}, {row['a[2]']:.3f}]  t0={row['t0']:.4f}  "
              f"sv={row['sv']:.4f}  sz={row['sz']:.4f}")
        print(f"  LL={ic['log_lik']:.1f}  AIC={ic['AIC']:.1f}  BIC={ic['BIC']:.1f}")

        # SRDM
        print(f"\n  --- SRDM (no lapse) ---")
        srdm_mle = fit_srdm(srdm_model, data)
        ic2 = aic_bic(srdm_mle, srdm_n_params)
        save_fit_row(SRDM_OUT, pid, N_LEVELS, actual_levels, srdm_mle, ic2,
                     extra={"t0_hi": data["t0_hi"]})
        row = srdm_mle.optimized_params_pd.iloc[0]
        print(f"  c=[{row['c[1]']:.3f}, {row['c[2]']:.3f}]  B={row['B']:.3f}  "
              f"r={row['r']:.2f}  t0={row['t0']:.4f}")
        print(f"  LL={ic2['log_lik']:.1f}  AIC={ic2['AIC']:.1f}  BIC={ic2['BIC']:.1f}")

        delta = ic["AIC"] - ic2["AIC"]
        print(f"\n  ΔAIC(DDM-SRDM) = {delta:+.1f}  ({'DDM' if delta<0 else 'SRDM'} wins)")

    print(f"\nAll fits written/updated in:\n  {DDM_OUT}\n  {SRDM_OUT}")


if __name__ == "__main__":
    main()
